# Mount ADLS Gen2 in Databricks

Mounts the `kalshi-data` container using OAuth with the service principal.

**Prerequisites:**
1. Secret scope `kalshi-secrets` linked to Azure Key Vault
2. Key Vault secret `sp-client-secret` (service principal client secret)
3. Service principal has **Storage Blob Data Contributor** on the storage account

**Get values from Azure CLI:**
```bash
az deployment group show -g rg-kalshi-pipeline -n kalshi-deploy --query properties.outputs.storageAccountName.value -o tsv
az account show --query tenantId -o tsv
```

**Add SP secret to Key Vault (one-time):**
```bash
az keyvault secret set --vault-name <keyVaultName> --name sp-client-secret --value "<SP_PASSWORD>"
```

In [ ]:
# Config - update STORAGE_ACCOUNT from: az deployment group show -g rg-kalshi-pipeline -n kalshi-deploy --query properties.outputs.storageAccountName.value -o tsv
STORAGE_ACCOUNT = ""  # e.g. stkalshiogihujuict7io from deployment outputs
TENANT_ID = "b4b203c1-e6ed-4319-a1aa-80694c9ce7e9"
CLIENT_ID = "5e532278-857e-493b-9185-ea6b714d1e42"  # sp-kalshi-databricks

In [ ]:
mount_point = "/mnt/kalshi-data"
container = "kalshi-data"

# Skip if already mounted
if any(m.mountPoint == mount_point for m in dbutils.fs.mounts()):
    print(f"Already mounted at {mount_point}")
else:
    if not STORAGE_ACCOUNT or not TENANT_ID or not CLIENT_ID:
        raise ValueError("Set STORAGE_ACCOUNT, TENANT_ID, and CLIENT_ID in the config cell above")
    
    client_secret = dbutils.secrets.get(scope="kalshi-secrets", key="sp-client-secret")
    
    configs = {
        "fs.azure.account.auth.type": "OAuth",
        "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        "fs.azure.account.oauth2.client.id": CLIENT_ID,
        "fs.azure.account.oauth2.client.secret": client_secret,
        "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
    }
    
    source = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    dbutils.fs.mount(source=source, mount_point=mount_point, extra_configs=configs)
    print(f"Mounted {source} at {mount_point}")

In [ ]:
# Verify - list root of mount
dbutils.fs.ls("/mnt/kalshi-data")